# How to view extracted DICOM headers for MIDRC images
---
**Author:** Christopher Meyer, PhD; Assistant Director of User Solutions and Scientific Support.<br>**Last updated**: January 28, 2026<br>

**Overview:** This notebook demonstrates how to access and review the DICOM headers for imaging studies / series in the MIDRC data commons. 

* First, a query will be used to select a subset of imaging series.
* Next, the DICOM headers for those imaging series will be accessed using the property `dicom_header_file_guid` in the data_file (or imaging_study) index.
* Finally, the downloaded DICOM header files will be concatenated into a dataframe for exploratory analysis.

## 1) Set up Python environment
---


### Download an API key file containing your credentials
---

1) Navigate to the MIDRC data portal in your browser: https://data.midrc.org.

2) Read and accept the DUA (if you haven't already).

3) Navigate to the user profile page: https://data.midrc.org/identity

4) Click on the button "Create API Key" and save the `credentials.json` file somewhere safe as "midrc-credentials.json".


### Set local variables
---
Change the following `cred` variable path to point to your credentials file downloaded from the MIDRC data portal following the instructions above.

In [1]:
# ## The MIDRC Data Commons (production environment)
# cred = "/Users/cgmeyer/Downloads/midrc-credentials.json" # location of your MIDRC credentials, downloaded from https://data.midrc.org/identity by clicking "Create API key" button and saving the credentials.json locally
# api = "https://data.midrc.org" # The base URL of the data commons being queried. This shouldn't change for MIDRC.


In [2]:
## MIDRC Staging Environment
scred = "/Users/cgmeyer/Downloads/midrc-staging-credentials.json" # location of your MIDRC credentials, downloaded from https://data.midrc.org/identity by clicking "Create API key" button and saving the credentials.json locally
sapi = "https://staging.midrc.org" # The base URL of the data commons being queried. This shouldn't change for MIDRC.


### Install / Import Python Packages and Scripts

In [3]:
## The packages below may be necessary for users to install according to the imports necessary in the subsequent cells.

import sys
#!{sys.executable} -m pip install
#!{sys.executable} -m pip install --upgrade pandas
#!{sys.executable} -m pip install --upgrade --ignore-installed PyYAML
#!{sys.executable} -m pip install --upgrade pip
# !{sys.executable} -m pip install --upgrade gen3
# !{sys.executable} -m pip install pydicom
#!{sys.executable} -m pip install --upgrade Pillow
#!{sys.executable} -m pip install psmpy
#!{sys.executable} -m pip install python-gdcm --upgrade
#!{sys.executable} -m pip install pylibjpeg --upgrade

In [4]:
## Import Python Packages and scripts

import os, subprocess
import pandas as pd
import numpy as np
import pydicom
from PIL import Image
import glob
import re
from pathlib import Path

# import some Gen3 packages
import gen3
from gen3.auth import Gen3Auth
from gen3.query import Gen3Query



### Initiate instances of the Gen3 SDK Classes using credentials file for authentication
---
Again, make sure the "cred" directory path variable reflects the location of your credentials file (path variables set above).

In [7]:
# ## The MIDRC Data Commons (production environment)
# auth = Gen3Auth(api, refresh_file=cred) # authentication class
# query = Gen3Query(auth) # query class


In [8]:
## MIDRC Staging Environment
sauth = Gen3Auth(sapi, refresh_file=scred) # authentication class
squery = Gen3Query(sauth) # query class


## 2) Build Cohorts by Sending Queries to the MIDRC APIs
#### General notes on sending queries:
* There are many ways to query and access metadata for cohort building in MIDRC, but this notebook will focus on using the [Gen3](https://gen3.org) graphQL query service ["guppy"](https://github.com/uc-cdis/guppy/#readme). This is the backend query service that [MIDRC's data explorer GUI](https://data.midrc.org/explorer) uses. So, anything you can do in the explorer GUI, you can do with guppy queries, and more!
* The guppy graphQL service has more functionality than is demonstrated in this simple example. You can find extensive documentation in GitHub [here](https://github.com/uc-cdis/guppy/blob/master/doc/queries.md) in case you'd like to build your own queries from scratch.
* The Gen3 SDK (intialized as `query` above in this notebook) has Python wrapper scripts to make sending queries to the guppy graphQL API simpler. The guppy SDK package can be viewed in GitHub [here](https://github.com/uc-cdis/gen3sdk-python/blob/master/gen3/query.py).
* Guppy queries focus on a particular type of data (cases, imaging studies, files, etc.), which corresponds to the major tabs in [MIDRC's data explorer GUI](https://data.midrc.org/explorer).
* Queries include arguments that are akin to selecting filter values in [MIDRC's data explorer GUI](https://data.midrc.org/explorer).
* To see more documentation about how to use and combine filters with various operator logic (like AND/OR/IN, etc.) see [this page](https://github.com/uc-cdis/guppy/blob/master/doc/queries.md#filter).

---


#### Set query parameters
---
* Here, we'll send a query to the `imaging_study` guppy index, which corresponds to the "Imaging Studies" tab of [MIDRC's data explorer GUI](https://data.midrc.org/explorer).
* The filters defined below can be modified to return different subsets of imaging studies. Here, we'll use rather restrictive parameters so the number of studies returned is small for demonstration purposes.
* If our query request is successful, the API response should be in JSON format, and it should contain a list of imaging study UIDs along with any other study-related data we ask for.


In [9]:
### Set some "imaging_study" query parameters

## Project ID: We're interested only in a particular dataset in MIDRC
project_id = "TCIA-RICORD_1a"

## Imaging study modality filter: we select only imaging studies with a modality of CT
study_modalities = ["CT"]

## Imaging study body part filter: here we select "chest" as the "LOINC system" filter, which is the body part examined
body_part_examined = "Chest"

## Case filters: we will select Hispanic males 70 years of age and older
sex = "Male"
age_threshold = 40

In [10]:
## Note: the "fields" option defines what fields we want the query to return. If set to "None", returns all available fields.

imaging_studies = squery.raw_data_download(
                    data_type="imaging_study",
                    fields=None,
                    filter_object={
                        "AND": [
                            {"=": {"project_id": project_id}},
                            {"=": {"loinc_system": body_part_examined}},
                            {"=": {"sex": sex}},
                            {">=": {"age_at_index": age_threshold}},
                            {"IN": {"study_modality": study_modalities}},
                        ]
                    },
                    sort_fields=[{"submitter_id": "asc"}]
                )

if len(imaging_studies) > 0:
    imaging_studies_ids = [i['submitter_id'] for i in imaging_studies if 'submitter_id' in i] ## make a list of the imaging study IDs returned
    print("Query returned {} study IDs from {} cases.".format(len(imaging_studies),len(set([i['case_ids'][0] for i in imaging_studies if 'case_ids' in i]))))
    print("Data is a list with rows like this:\n\t {}".format(imaging_studies[0:1]))
else:
    print("Your query returned no data! Please, check that query parameters are valid.")

Query returned 21 study IDs from 17 cases.
Data is a list with rows like this:
	 [{'_imaging_study_id': '9f4ded5f-4db0-4be6-9a87-576c9f068799', 'project_id': 'TCIA-RICORD_1a', 'submitter_id': '1.2.826.0.1.3680043.10.474.440808.3022', 'case_ids': ['440808-000005'], 'age_at_imaging': 60, 'loinc_code': '79077-4', 'loinc_contrast': 'W', 'loinc_long_common_name': 'CTA Pulmonary arteries for pulmonary embolus W contrast IV', 'loinc_method': 'CT.angio', 'loinc_system': 'Chest', 'study_description': 'THORAX PE', 'study_modality': ['CT'], 'study_year': 2004, 'study_year_shifted': 'true', 'study_uid': '1.2.826.0.1.3680043.10.474.440808.3022', 'sex': ['Male'], 'race': [], 'age_at_index': [60], 'index_event': [], 'zip': [], 'covid19_positive': ['Yes'], 'ethnicity': [], 'dataset_submitter_id': ['RICORD_1a'], 'license': ['https://creativecommons.org/licenses/by-nc/4.0/'], 'data_url_doi': ['https://doi.org/10.7937/VTW4-X588'], 'data_contributor': ['TCIA'], '_mr_series_file_count': 0, '_ct_series_file

In [11]:
imaging_studies_df = pd.DataFrame(imaging_studies)
display(imaging_studies_df)

,_imaging_study_id,project_id,submitter_id,case_ids,age_at_imaging,loinc_code,loinc_contrast,loinc_long_common_name,loinc_method,loinc_system,...,data_category,dicom_header_file_guid,data_file_source_node,data_file_annotation_name,data_file_annotation_method,data_file_image_data_modified,data_file_image_data_modification_name,data_file_image_data_modification_method,auth_resource_path,body_part_examined
0,9f4ded5f-4db0-4be6-9a87-576c9f068799,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.440808.3022,[440808-000005],60,79077-4,W,CTA Pulmonary arteries for pulmonary embolus W...,CT.angio,Chest,...,[CT],[dg.MD1R/35dce470-827b-46f4-9cad-dd7c70b822e1],[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,NaN
1,8f97ce47-fafa-4556-bb24-50dd514cfa35,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.440808.3616,[440808-000017],70,79077-4,W,CTA Pulmonary arteries for pulmonary embolus W...,CT.angio,Chest,...,[CT],[dg.MD1R/9f60ccb8-5580-4113-a38c-d5010d1f59ed],[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,NaN
2,ada2af8e-5be3-4f83-8594-3638d442e13d,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.440808.811,[440808-000016],60,79077-4,W,CTA Pulmonary arteries for pulmonary embolus W...,CT.angio,Chest,...,[CT],[dg.MD1R/aad0f25a-7bc9-4f92-8431-de424fca49af],[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,[CHEST]
3,4d5da3fe-5f69-4c36-adcc-8ac158083d6e,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.419639.308001698749...,[419639-000421],63,29252-4,WO,CT Chest WO contrast,CT,Chest,...,[CT],"[dg.MD1R/b9f3b074-f286-433c-9537-ade2ce122f4f,...",[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,[CHEST]
4,9da06350-31b8-485b-932e-fcee2e6db243,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.419639.198191490979...,[419639-001476],79,24628-0,W,CT Chest W contrast IV,CT,Chest,...,[CT],"[dg.MD1R/90aae19c-48f0-4a47-9278-56ba3dfb30e8,...",[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,NaN
5,11890e1d-cedb-4062-a679-ac68cf81a15c,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.419639.317104088349...,[419639-000582],63,24628-0,W,CT Chest W contrast IV,CT,Chest,...,[CT],"[dg.MD1R/94675c63-f185-49e8-be90-53ee82e81bdd,...",[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,NaN
6,c808d05c-6ae1-47a5-b1e5-5927121c0e5e,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.419639.290296058041...,[419639-000906],62,29252-4,WO,CT Chest WO contrast,CT,Chest,...,[CT],"[dg.MD1R/dff2daed-0e81-4b2c-af94-d701a445a3cf,...",[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,NaN
7,72fc18f5-859e-4809-9010-8abfdd881aa3,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.440808.2746,[440808-000008],65,79077-4,W,CTA Pulmonary arteries for pulmonary embolus W...,CT.angio,Chest,...,[CT],[dg.MD1R/7da0b971-a67e-46e1-b1b7-c3e78b840f8f],[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,NaN
8,15518781-2e0b-4f0d-a16b-60248b8e09af,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.419639.152757649181...,[419639-001012],79,29252-4,WO,CT Chest WO contrast,CT,Chest,...,[CT],"[dg.MD1R/ec47f591-249a-480b-b9d6-4a971d6826f4,...",[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,NaN
9,1ce75c3c-a0ef-45d6-938c-68be431d8cce,TCIA-RICORD_1a,1.2.826.0.1.3680043.10.474.440808.929,[440808-000011],65,79077-4,W,CTA Pulmonary arteries for pulmonary embolus W...,CT.angio,Chest,...,[CT],[dg.MD1R/89021007-a836-41ab-bb1c-372014af8a73],[ct_series_file],[],[],[],[],[],/programs/TCIA/projects/RICORD_1a,[CHEST]


### Create a download manifest for the DICOM header files
---
Using the results of our query, we can now access imaging study data for our selected cohort:
* CT images using `object_id`,
* **Only the DICOM headers** for the CT images using `dicom_header_file_guid`.

In [12]:
imaging_studies_df[['object_id','dicom_header_file_guid']]

,object_id,dicom_header_file_guid
0,[dg.MD1R/b07372d0-58d6-4d9a-8092-e2ec1e8a9a25],[dg.MD1R/35dce470-827b-46f4-9cad-dd7c70b822e1]
1,[dg.MD1R/01605965-bcb6-4a95-8af5-291e019e72f7],[dg.MD1R/9f60ccb8-5580-4113-a38c-d5010d1f59ed]
2,[dg.MD1R/6ed2e60c-0095-412a-a760-2a5841c18d14],[dg.MD1R/aad0f25a-7bc9-4f92-8431-de424fca49af]
3,"[dg.MD1R/f7209dde-5e18-4bb5-bec9-4a113ee4f18d,...","[dg.MD1R/b9f3b074-f286-433c-9537-ade2ce122f4f,..."
4,"[dg.MD1R/17e18624-48f3-481c-8eb5-ce662398ebfd,...","[dg.MD1R/90aae19c-48f0-4a47-9278-56ba3dfb30e8,..."
5,"[dg.MD1R/a208b67e-f4b4-4f5a-9af0-13d86b2753d3,...","[dg.MD1R/94675c63-f185-49e8-be90-53ee82e81bdd,..."
6,"[dg.MD1R/4d067c79-d370-466b-9962-d130d58e9237,...","[dg.MD1R/dff2daed-0e81-4b2c-af94-d701a445a3cf,..."
7,[dg.MD1R/81b7bf10-4c67-4585-a8f7-f42076b3e9e5],[dg.MD1R/7da0b971-a67e-46e1-b1b7-c3e78b840f8f]
8,"[dg.MD1R/64a4b281-b1a9-4c05-b4a8-0597e8ca0abb,...","[dg.MD1R/ec47f591-249a-480b-b9d6-4a971d6826f4,..."
9,[dg.MD1R/d7a0311a-edb5-45a6-ae02-db500da40e1f],[dg.MD1R/89021007-a836-41ab-bb1c-372014af8a73]


In [13]:
## Flatten the column of header GUID lists for each imaging study into a single list of GUIDs
#list(imaging_studies_df['dicom_header_file_guid']))
header_guids = imaging_studies_df['dicom_header_file_guid'].apply(list).sum()
print(f"Creating a file download manifest for {len(header_guids)} DICOM header files for {len(imaging_studies_df)} imaging studies.")

Creating a file download manifest for 51 DICOM header files for 21 imaging studies.


In [15]:
## Define / Use a function to write the manifest file

def write_manifest(guids, filename="DICOM_header_file_manifest.json", output_dir="."):
        manifest_file = f"{output_dir}/{filename}"
        with open(manifest_file, "w") as manifest:
            manifest.write("[\n  {\n")
            count = 0
            for guid in guids:
                count += 1
                file_line = '    "object_id": "{}"\n'.format(guid)
                manifest.write(file_line)
                if count == len(guids):
                    manifest.write("  }]")
                else:
                    manifest.write("  },\n  {\n")
        print(f"File download manifest written to:\n\t{manifest_file}")
        return manifest_file

manifest_file = write_manifest(header_guids)

File download manifest written to:
	./DICOM_header_file_manifest.json


## 4) Access DICOM header files using their object_id / data GUID (globally unique identifiers)
---
Now that we have a list of DICOM header file GUIDs we want to download, we can use either the gen3-client or the gen3 SDK to download the files. 

For instructions on how to install and use the gen3-client, please see [the MIDRC quick-start guide](https://data.midrc.org/dashboard/Public/documentation/Gen3_MIDRC_GetStarted.pdf), which can be found linked here and in the MIDRC data portal header as "Get Started".

Below we use the gen3 SDK command `gen3 drs-pull object` which is [documented in detail here](https://github.com/uc-cdis/gen3sdk-python/blob/master/docs/howto/drsDownloading.md).

### Use the Gen3 SDK command `gen3 drs-pull object` to download an individual file

In [17]:
## Make a new directory in Colab /content dir for downloaded files
## Note: if this command is run in Google Colab, this will not alter any local directories
os.system("rm -r dicom_header_downloads")
os.system("mkdir -p dicom_header_downloads")


rm: dicom_header_downloads: No such file or directory


0

In [18]:
# ## We can use a simple loop to download all files and keep track of successes and failures
# ## Here we will only download one image to save time for demo purposes
# oid_num = 1
# success,failure,other=[],[],[]
# count,total = 0,len(object_ids[0:oid_num])
# for object_id in object_ids[0:oid_num]:
#     count+=1
#     cmd = "gen3 --auth {} --endpoint data.midrc.org drs-pull object {} --output-dir dicom_header_downloads".format(cred,object_id)
#     stout = subprocess.run(cmd, shell=True, capture_output=True)
#     if not stout.stdout:
#         raise Exception(f"gen3 sdk failure: {stout.stderr}")
#     #print("Progress ({}/{}): {}".format(count,total,stout.stdout))
#     print("Progress ({}/{}): {}".format(count,total,stout.stdout.decode("utf-8")))
#     if "failed" in str(stout.stdout):
#         failure.append(object_id)
#     elif "successfully" in str(stout.stdout):
#         success.append(object_id)
#     else:
#         other.append(object_id)


In [19]:
## Configure gen3-client

cmd = f"gen3-client configure --profile=midrcstaging --apiendpoint={sapi} --cred={scred}"
stout = subprocess.run(cmd, shell=True, capture_output=True)
if not stout.stderr:
    raise Exception(f"gen3 sdk failure: {stout.stderr}")
#print("Progress ({}/{}): {}".format(count,total,stout.stdout))
print(f"{stout.stderr.decode('utf-8')}")

2026/01/28 14:10:07 A new version of gen3-client is available! The latest version is 2026.2.0. You are using version 2025.07
2026/01/28 14:10:07 Please download the latest gen3-client release from https://github.com/uc-cdis/cdis-data-client/releases/latest
2026/01/28 14:10:08 Profile 'midrcstaging' has been configured successfully.



In [20]:
## We can use a simple loop to download all files and keep track of successes and failures
## Here we will only download one image to save time for demo purposes
download_path = 'dicom_header_downloads'
cmd = f"gen3-client download-multiple --profile=midrcstaging --manifest={manifest_file} --download-path={download_path} --numparallel=3 --no-prompt"
stout = subprocess.run(cmd, shell=True, capture_output=True)
if not stout.stderr:
    raise Exception(f"gen3 sdk failure: {stout.stderr}")
print(f"{stout.stderr.decode('utf-8')}")
# if "successfully" in str(stout.stdout):
#     success.append(object_id)
# else:
#     failure.append(object_id)


2026/01/28 14:10:10 A new version of gen3-client is available! The latest version is 2026.2.0. You are using version 2025.07
2026/01/28 14:10:10 Please download the latest gen3-client release from https://github.com/uc-cdis/cdis-data-client/releases/latest
2026/01/28 14:10:10 Reading manifest...
2026/01/28 14:10:10 Total number of objects in manifest: 51
2026/01/28 14:10:10 Preparing file info for each file, please wait...
2026/01/28 14:10:15 File info prepared successfully
2026/01/28 14:10:36 51 files downloaded.



In [23]:
# Get a list of all downloaded .dcm files
header_zips = glob.glob(pathname=f'{download_path}/**/*_headers.zip',recursive=True,)
len(header_zips)

51

In [44]:
study_dir = Path(zip).parent.absolute()
study_dir

PosixPath('/Users/cgmeyer/Documents/Notes/MIDRC/dicom_headers/notebooks/dicom_header_downloads/TCIA-RICORD_1a/419639-001596/1.2.826.0.1.3680043.10.474.419639.288267420128264959058930883141')

In [47]:
series_uid

'1.2.826.0.1.3680043.10.474.419639.217659038521342318971586744906'

In [25]:
zip = header_zips[0]

In [26]:
print(zip)
print(download_path)

dicom_header_downloads/TCIA-RICORD_1a/419639-001596/1.2.826.0.1.3680043.10.474.419639.288267420128264959058930883141/1.2.826.0.1.3680043.10.474.419639.217659038521342318971586744906_headers.zip
dicom_header_downloads


In [49]:
for i in range(0,len(header_zips)):
    zip = header_zips[i]
    study_dir = Path(zip).parent.absolute()
    series_uid = Path(zip).stem.replace('_headers','')    
    print(f"({i+1}/{len(header_zips)}): {series_uid}")
    dcms = sorted(glob.glob(f"{study_dir}/{series_uid}/*.dcm"))
    if len(dcms) == 0: # unzip if not already unzipped
        print(f"({i+1}/{len(header_zips)}): Unzipping {series_uid}")
        os.system(f"unzip -q -o {zip} -d {study_dir}")
        

(1/51): 1.2.826.0.1.3680043.10.474.419639.217659038521342318971586744906
(1/51): Unzipping 1.2.826.0.1.3680043.10.474.419639.217659038521342318971586744906
(2/51): 1.2.826.0.1.3680043.10.474.419639.222779076385397231259911522028
(2/51): Unzipping 1.2.826.0.1.3680043.10.474.419639.222779076385397231259911522028
(3/51): 1.2.826.0.1.3680043.10.474.419639.248789328460452579567145413428
(3/51): Unzipping 1.2.826.0.1.3680043.10.474.419639.248789328460452579567145413428
(4/51): 1.2.826.0.1.3680043.10.474.419639.433800011920523157764565727810
(4/51): Unzipping 1.2.826.0.1.3680043.10.474.419639.433800011920523157764565727810
(5/51): 1.2.826.0.1.3680043.10.474.419639.148938292763592847048276595683
(5/51): Unzipping 1.2.826.0.1.3680043.10.474.419639.148938292763592847048276595683
(6/51): 1.2.826.0.1.3680043.10.474.440808.1783
(6/51): Unzipping 1.2.826.0.1.3680043.10.474.440808.1783
(7/51): 1.2.826.0.1.3680043.10.474.440808.1139
(7/51): Unzipping 1.2.826.0.1.3680043.10.474.440808.1139
(8/51): 1.2.

### Explore the DICOM Headers
---
Note that some of the files may contain compressed pixel data that require other packages to view; so, for this demo we'll simply skip over those using the following loop.

In [29]:
ds = pydicom.dcmread(image_files[0],force=True)
display(ds)

NameError: name 'image_files' is not defined

In [30]:
# Access individual elements
display(ds.file_meta)
display(ds.ImageType)
display(ds[0x0008, 0x0016])


NameError: name 'ds' is not defined

In [31]:
# View the dicom metadata for all files as a DataFrame
dfs = []
for image_file in image_files:
    ds = pydicom.dcmread(image_file)
    df = pd.DataFrame(ds.values())
    df[0] = df[0].apply(lambda x: pydicom.dataelem.convert_raw_data_element(x) if isinstance(x, pydicom.dataelem.RawDataElement) else x)
    df['name'] = df[0].apply(lambda x: x.name)
    df['value'] = df[0].apply(lambda x: x.value)
    df = df[['name', 'value']]
    df = df.set_index('name').T.reset_index(drop=True)
    df['filename'] = image_file
    df.drop(columns=['Pixel Data'],inplace=True) # drop the pixel data as it's too large and nonsensical to store in a DataFrame
    dfs.append(df)

NameError: name 'image_files' is not defined

In [32]:
# Make a master dataframe for all images using only headers in all dataframes
headers = list(set.intersection(*map(set,dfs)))
df = pd.concat([df[headers] for df in dfs])
df.set_index('filename',inplace=True)


TypeError: unbound method set.intersection() needs an argument

In [33]:
display(df)

NameError: name 'df' is not defined

In [34]:
## Export the file metadata as a TSV file
filename = "MIDRC_DICOM_metadata.tsv"
df.to_csv(filename, sep='\t')


NameError: name 'df' is not defined